<a href="https://colab.research.google.com/github/almendraapolaya/DI_Bootcamp_a/blob/main/Week_7/Day_3%20/Daily_challenge/Daily_challenge_day3_w7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Daily Challenge :
===
**Preprocess & fine-tune transformer-based models**

What you will create?

By the end of this challenge, you will have a fine-tuned transformer model (BERT or XLM-RoBERTa) capable of classifying text into different categories. Additionally, you will structure the data for training, validate it using cross-validation, and understand how to optimize these models for better performance.

**Task 1:**



**Understanding BERT and XLM-RoBERTa**

Objective: Learn how transformer models work and their role in NLP tasks.

Instructions:

- Read through the descriptions of BERT and XLM-RoBERTa.
- Understand how these models process text using tokenization.
- Learn about different pre-trained versions of these models and their characteristics.


**BERT (Bidirectional Encoder Representations from Transformers):** It reads text in both directions simultaneously to understand context. However, standard BERT is primarily trained on English.

**XLM-RoBERTa:** This is a multilingual version of RoBERTa. Since your dataset includes French, Thai, Arabic, Urdu, and more, XLM-RoBERTa is actually the superior choice here because it was pre-trained on 100 different languages.

**Understand how these models process text using tokenization**

In BERT-style models, text isn't just split into words; it's split into sub-words using a method called Byte-Pair Encoding or WordPiece. This prevents "Out of Vocabulary" errors.

In [ ]:
!pip install transformers[torch] datasets

In [ ]:
import pandas as pd
from transformers import BertTokenizer, XLMRobertaTokenizer

df = pd.read_csv('train.csv')

bert_tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')
xlm_tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')

sample_premise = df.iloc[0]['premise']
sample_hypothesis = df.iloc[0]['hypothesis']

print(f"Original Premise: {sample_premise}")
print(f"Original Hypothesis: {sample_hypothesis}\n")

#BERT Tokenization
bert_input = bert_tokenizer(sample_premise, sample_hypothesis)
print("--- BERT Tokens ---")
print(bert_tokenizer.convert_ids_to_tokens(bert_input['input_ids']))

#XLM-RoBERTa Tokenization
xlm_input = xlm_tokenizer(sample_premise, sample_hypothesis)
print("\n--- XLM-RoBERTa Tokens ---")
print(xlm_tokenizer.convert_ids_to_tokens(xlm_input['input_ids']))

**Task 2**:

**Tokenizing Text**

Objective: Understand how to tokenize text using pre-trained tokenizers.

Instructions:

- Use the BertTokenizer and XLMRobertaTokenizer to convert sentences into tokenized input.
- Explore the different token types, such as input_ids, attention_mask, and labels.
- Experiment with single-sentence and two-sentence tokenization.

In [ ]:
import torch
from transformers import AutoTokenizer

model_name = 'bert-base-multilingual-cased'
tokenizer = AutoTokenizer.from_pretrained(model_name)

premise = "The rules were put together with these comments in mind."
hypothesis = "The rules were developed recently."

encoded_dict = tokenizer(
    text=premise,
    text_pair=hypothesis,
    add_special_tokens=True,
    max_length=32,
    padding='max_length',
    truncation=True,
    return_attention_mask=True,
    return_tensors='pt'
)

# Results:
print("Successfully created keys:", encoded_dict.keys())

print("\n--- 1. Input IDs ---")
print(encoded_dict['input_ids'])

print("\n--- 2. Attention Mask ---")
print(encoded_dict['attention_mask'])

print("\n--- 3. Decoded Sequence ---")
print(tokenizer.decode(encoded_dict['input_ids'][0]))

**Task 3:**

**Preparing Input Data for the Model
Objective:** Format input data correctly for transformer models.

Instructions:

- Ensure that input sentences are padded and possibly truncated to max_length.
- Understand and set special tokens such as <s> and </s>.
- Learn about attention_mask and how it helps the model ignore padding tokens.

Functions to use:

tokenizer.encode_plus()
tokenizer.special_tokens_map
tokenizer.vocab_size

In [ ]:
print(f"Special Tokens Map: {tokenizer.special_tokens_map}")
print(f"Vocabulary Size: {tokenizer.vocab_size}")

# To check for <s> and </s> :
print(f"CLS token: {tokenizer.cls_token} (ID: {tokenizer.cls_token_id})")
print(f"SEP token: {tokenizer.sep_token} (ID: {tokenizer.sep_token_id})")

In [ ]:
import torch
from torch.utils.data import Dataset

class NliDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        premise = str(self.df.iloc[index]['premise'])
        hypothesis = str(self.df.iloc[index]['hypothesis'])
        label = self.df.iloc[index]['label']

        encoding = self.tokenizer(
            premise,
            hypothesis,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

train_dataset = NliDataset(df, tokenizer)

# Test:
sample_item = train_dataset[0]
print("Keys in one data item:", sample_item.keys())
print("Shape of input_ids:", sample_item['input_ids'].shape)

In [ ]:
# Final check for Task 3 requirements:
print("--- Task 3 Metadata Check ---")

print(f"Special Tokens Map: {tokenizer.special_tokens_map}")

# Vocabulary Size:
print(f"Tokenizer Vocab Size: {tokenizer.vocab_size}")

# Attention Mask logic:
sample_item = train_dataset[0]
print("\n--- Attention Mask Verification ---")
print(f"Input IDs (first 15): {sample_item['input_ids'][:15]}")
print(f"Mask IDs (first 15):  {sample_item['attention_mask'][:15]}")


**Learn about attention_mask and how it helps the model ignore padding tokens:**

- A '1' in the mask means the model processes that token.
- A '0' (usually at the end of the tensor) tells the model to ignore that padding.

**Task 4:**

**Loading and Exploring the Dataset
Objective:** Load the dataset and explore its structure.

Instructions:

- Load the training and testing data from CSV files.
- Display the first few rows to understand its structure.
- Identify the columns needed for training the model.

Functions to use:

pd.read_csv()
df.head()
df.shape

In [ ]:
import pandas as pd

df = pd.read_csv('train.csv')

print(f"Dataset Shape: {df.shape}")

print("\n--- First 5 Rows ---")
print(df.head())

# Columns needed for training:
cols_for_training = ['premise', 'hypothesis', 'label']
print(f"\nTarget Training Columns: {cols_for_training}")

In [ ]:
import pandas as pd

# Loading both files:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# Final Check: Display first rows and shapes:
print(f"Train Shape: {train_df.shape} | Test Shape: {test_df.shape}")
print("\n--- Train Columns ---")
print(train_df.columns.tolist())
print("\n--- Test Columns ---")
print(test_df.columns.tolist())

**Task 5:**

**Creating Cross-Validation Folds
Objective:** Implement k-fold cross-validation for training.

Instructions:

- Use StratifiedKFold to create 5 training-validation splits.
- Ensure that each fold maintains the same label distribution.
- Store the training and validation splits in separate lists.

Functions to use:

from sklearn.model_selection import StratifiedKFold
kf.split()
StratifiedKFold(shuffle=True)

In [ ]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Lists to store splits:
train_folds = []
val_folds = []

# Creating the splits:
for train_index, val_index in skf.split(train_df, train_df['label']):
    train_folds.append(train_df.iloc[train_index])
    val_folds.append(train_df.iloc[val_index])

# Task 5:
print(f"Number of folds created: {len(train_folds)}")
print(f"First fold training size: {len(train_folds[0])}")
print(f"First fold validation size: {len(val_folds[0])}")

print("\nLabel distribution in Validation Fold 1:")
print(val_folds[0]['label'].value_counts(normalize=True))

**Conclusion & Insights:**

This challenge demonstrates the end-to-end workflow of adapting 'state of the art' Transformer models for a multilingual Natural Language Inference (NLI) task. By comparing BERT and XLM-RoBERTa, we observed how tokenization strategies like WordPiece and SentencePiece handle linguistic nuances and special tokens ([SEP] vs </s>) differently. The successful implementation of the NliDataset class highlights the importance of precise data formatting specifically padding, truncation, and attention masking to ensure that the model correctly prioritizes meaningful text over structural filler during the training process.

A key insight from the exploratory phase was the balanced distribution of the entailment, neutral, and contradiction labels. This balance, coupled with the use of StratifiedKFold, ensures that each training cycle is exposed to a representative slice of the ground truth, preventing the model from developing a bias toward any specific class. Using five folds allows for a robust evaluation of the model's stability, providing a more reliable estimate of performance than a simple train-test split, especially when dealing with the diverse linguistic patterns found in train.csv.

Ultimately, the choice of XLM-RoBERTa as the primary architecture is justified by the multilingual nature of the dataset. Unlike standard monolingual models, XLM-RoBERTa’s cross-lingual pre-training allows it to share semantic knowledge across different languages, such as English, French, and Thai, within the same embedding space. This approach not only streamlines the training pipeline but also maximizes the model's ability to generalize across the various languages present in the test set, leading to more accurate and reliable text classification.